# q=20 Bessel z-scan — full radial + angular phase retrieval

This is the **canonical experimental aberration-retrieval notebook**. It replaces the old `Recovered phase at z0` / `Predicted XZ/YZ before-after correction` path.

The correction is one **transverse** residual phase. The measured z-stack supplies radial diversity: each z plane constrains a different input annulus. Longitudinal evolution is produced by propagation; no independent longitudinal correction phase is fitted.

The implementation follows Miao *et al.*, *Optics Express* **30**, 11360–11371 (2022), DOI 10.1364/OE.454796.


In [ ]:
from pathlib import Path
import json, os, sys
from IPython.display import Image, display

HERE = Path.cwd().resolve()
if not (HERE / 'run_miao_full_q20.py').is_file():
    candidate = HERE / 'notebooks' / 'experimental' / 'axicon_aberration_correction'
    if candidate.is_dir(): HERE = candidate
if str(HERE) not in sys.path: sys.path.insert(0, str(HERE))
DATA_DIR = Path(os.environ.get('BESSEL_ZSCAN_DATA_DIR', HERE / 'z-scan 2 1010')).expanduser().resolve()
OUTPUT_DIR = HERE / 'outputs' / 'miao_full_q20'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Data:', DATA_DIR)
print('Outputs:', OUTPUT_DIR)


## Physics being solved

At every measured z plane the algorithm retrieves the optimum transverse wavenumber $k_\perp^{opt}$ and the angular Bessel-mode coefficients. The radial residual follows

$$\frac{d\psi_\rho}{d\rho}=k\tan\alpha-k_\perp^{opt},\qquad \rho_z=\frac{z k_\perp^{opt}}{k\tan\alpha}.$$

The radial gradient is integrated and combined with the angular modal residual. The programmed $q\theta$ phase is already the target beam topology and is **not** part of the correction.

By default $m=\pm1$ is excluded because these camera planes are morphology-centred; quantitative coma/pointing retrieval requires a calibrated optical axis.


In [ ]:
from run_miao_full_q20 import run_miao_full_q20

metrics, summary = run_miao_full_q20(
    DATA_DIR, OUTPUT_DIR,
    q=20, wavelength_m=1030e-9, pixel_pitch_m=5.5e-6,
    # Supply these two values when independently calibrated:
    nominal_k_tan_alpha_m_inv=None,
    absolute_z_at_relative_zero_mm=None,
    include_first_order=False,
)
print(json.dumps(summary, indent=2))
display(metrics)


## 1. Radial physics from $k_\perp(z)$

This is the part missing from the earlier fixed-$k_\perp$ retrieval. A z-dependent optimum transverse wavenumber carries the radial phase-gradient information. If the nominal cone angle is not independently calibrated, the code reports the radial phase relative to the median fitted cone rather than pretending the absolute radial slope is known.


In [ ]:
display(Image(filename=str(OUTPUT_DIR / '01_miao_kperp_and_radial_phase.png')))


## 2. Full transverse residual wavefront

The residual combines the integrated radial phase with the retrieved angular field. It should **not** show the old 20-spoke target-vortex wrapping merely because q=20; the programmed vortex has been factored out. Wrapped phase is interpolated through the complex field, not as a scalar angle.


In [ ]:
display(Image(filename=str(OUTPUT_DIR / '02_miao_full_transverse_residual.png')))


## 3. One transverse field propagated through the whole z range

The longitudinal comparison below is generated by propagating one transverse field. The measured column is core-centred **morphology**, not absolute beam trajectory. The final model column is a counterfactual in which the inferred phase is conjugated; it is not post-SLM experimental data.


In [ ]:
display(Image(filename=str(OUTPUT_DIR / '03_miao_single_transverse_field_xz_yz.png')))
display(Image(filename=str(OUTPUT_DIR / '04_miao_heldout_forward_metrics.png')))


## Claim boundary

A retrieved wavefront is accepted as an explanatory phase model only if it improves **held-out** z-plane agreement over the nominal beam. The conjugate mask remains a model-derived candidate. Hardware application still requires camera/input ↔ SLM2 magnification, rotation/parity, illuminated footprint, 1030-nm phase LUT, and then a fresh measured post-correction z-stack.

The Bessel intensity inversion also has the conjugate / π-rotation ambiguity described by Miao *et al.*; without an independent input-plane orientation/intensity measurement, this notebook reports that ambiguity as unresolved rather than silently selecting one branch.
